In [1]:
# Import libraries
import pandas as pd
import os 
import sys

sys.path.append(os.path.abspath(".."))

import functions.wrangling as wrg

# Set working directory
os.chdir(r"G:\.shortcut-targets-by-id\1qO0AfYMqzVbXreDMm-gZUvrYXtVZCnDA\CHL8010F2  CPCSSN Dataset") 

In [2]:
# Load and clean datasets

# Define and load file paths 
file_paths = {
    'patient': 'C4MPatient.csv',
    'lab': 'C4MLab.csv',
    'diag': 'C4MEncounterdiagnosis.csv',
    'condition': 'C4MHealthCondition.csv'
}
datasets = wrg.load_csv(file_paths)

# Specify columns to keep from each dataset
columns_to_keep = {
    'patient': ["Patient_ID", "Sex", "BirthYear"],
    'lab': ["Patient_ID", "Name_calc", "TestResult_calc", "PerformedDate"],
    'diag': ["Patient_ID", "DiagnosisText_calc", "DiagnosisCode_calc", "DateCreated"],
    'condition': ["Patient_ID", "DiagnosisText_calc", "DateCreated"]
}
datasets = wrg.select_columns(datasets, columns_to_keep)

# Clean diagnosis data 
diagnosis_cleaning_steps = [
    ('DiagnosisText_calc', 'uppercase'),
    ('DiagnosisCode_calc', 'strip'),
    ('DiagnosisCode_calc', 'dropna'),
    ('DateCreated', 'datetime')
]
datasets['diag'] = wrg.replace_string_nan(datasets['diag'], 'DiagnosisCode_calc')
datasets['diag'] = wrg.preprocess_data(datasets['diag'], diagnosis_cleaning_steps)

In [3]:
# Extract bipolar disorder lab results and handle missing cases

# Define BD ICD-9 codes and relevant markers
bd_codes = ["296.0", "296.1", "296.4", "296.5", "296.6", "296.7", "296.80", "296.89"]
relevant_markers = ["TOTAL CHOLESTEROL", "HBA1C", "HDL", "FASTING GLUCOSE", "LDL", "INR", "GLUCOSE TOLERANCE"]

# Extract lab results that occur after BD diagnosis (filtered by relevant markers)
bd_labs_after = wrg.extract_labs_relative_to_diagnosis(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    lab_test_names=relevant_markers
)

# Get the first BD diagnosis date per patient
first_dx = wrg.get_first_matching_diagnosis(
    df=datasets['diag'],
    diagnosis_col='DiagnosisCode_calc',
    date_col='DateCreated',
    target_codes=bd_codes,
    new_date_col='BD_Diagnosis_Date',
    new_code_col='BD_Code'
)

# Merge diagnosis info into labs
bd_labs_after = bd_labs_after.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date', 'BD_Code']],
    on='Patient_ID', how='left'
).drop(columns=['Lab_Timing'])

# Pivot lab data to wide format for same-day comorbidity + lab summary
bd_labs_after_wide = wrg.pivot_lab_data(
    bd_labs_after,
    index_cols=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    name_col='Name_calc',
    value_col='TestResult_calc'
).sort_values(['Patient_ID', 'PerformedDate'])

# Classify lab timing
lab_timing_summary_relevant, bd_first_clean = wrg.classify_lab_timing(
    lab_df=datasets['lab'][datasets['lab']['Name_calc'].isin(relevant_markers)],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    relevant_markers=None
)

# Count patients with same-day non-BD diagnosis
non_bd_same_day_count = wrg.other_dx_same_day(
    diag_df=datasets['diag'],
    bd_first_clean=bd_first_clean,
    bd_codes=bd_codes
)

# Count patients with labs before, after, or both
lab_groups = wrg.split_patients_by_lab_timing(lab_timing_summary_relevant)
only_before = wrg.ensure_patient_id_column(lab_groups['only_before'])
only_after = wrg.ensure_patient_id_column(lab_groups['only_after'])
both = wrg.ensure_patient_id_column(lab_groups['both'])

# Patients with labs AFTER BD diagnosis AND non-BD diagnosis on same day
nonbd_diag_same_day_after = wrg.non_bd_same_day_with_labs_after(
    diag_df=datasets['diag'],
    bd_labs_after_df=bd_labs_after_wide,
    bd_first_clean=bd_first_clean,
    bd_codes=bd_codes
)

# Filter labs to those within 2 years of BD diagnosis
bd_year_labs_after = wrg.filter_labs_within_window(
    df=bd_labs_after,
    date_col='PerformedDate',
    ref_date_col='BD_Diagnosis_Date',
    window_days=730
)

# Drop same-day duplicate entries
bd_year_labs_after = (
    bd_year_labs_after.sort_values(by=['Patient_ID', 'PerformedDate', 'Name_calc', 'TestResult_calc'], na_position='last')
    .drop_duplicates(subset=['Patient_ID', 'PerformedDate', 'Name_calc'], keep='first')
)

# Check for repeated tests on same day, such as multiple entries for the same test
dup_tests_same_day = (
    bd_year_labs_after.groupby(['Patient_ID', 'PerformedDate', 'Name_calc'])
    .size()
    .reset_index(name='n')
)
dup_tests_same_day = dup_tests_same_day[dup_tests_same_day['n'] > 1]
patients_with_dup_tests = dup_tests_same_day['Patient_ID'].nunique()

# Pivot to wide format
bd_labs_within_2yrs = wrg.pivot_lab_data(
    df=bd_year_labs_after,
    index_cols=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    name_col='Name_calc',
    value_col='TestResult_calc'
).sort_values(['Patient_ID', 'PerformedDate'])

# Add age and sex data for each patient
bd_labs_within_2yrs = wrg.add_demographics_to_labs(
    lab_df=bd_labs_within_2yrs,
    patient_df=datasets['patient']
)

# Identify patients with comorbidities (non-BD diagnosis on or after BD diagnosis date)
patients_with_comorbidity = wrg.label_comorbidity(
    diag_df=datasets['diag'],
    bd_dx_df=first_dx,
    target_codes=bd_codes,
    patient_col='Patient_ID',
    code_col='DiagnosisCode_calc',
    date_col='DateCreated',
    bd_date_col='BD_Diagnosis_Date'
)

# Label patients with comorbidity
bd_labs_within_2yrs['Comorbidity'] = bd_labs_within_2yrs['Patient_ID'].apply(
    lambda x: 1 if x in patients_with_comorbidity else 0
)

summary_stats = [
    ("All patients have BD as their first-ever diagnosis", bd_first_clean['first_any_dx_code'].isin(bd_codes).all()),
    ("Total BD patients (first-ever diagnosis)", bd_first_clean['Patient_ID'].nunique()),
    ("Patients with ANY lab results", lab_timing_summary_relevant['Patient_ID'].nunique()),
    ("Patients with labs BEFORE diagnosis", pd.concat([only_before, both])['Patient_ID'].nunique()),
    ("Patients with labs AFTER diagnosis", pd.concat([only_after, both])['Patient_ID'].nunique()),
    ("Patients with labs BOTH before and after diagnosis", both['Patient_ID'].nunique()),
    ("Number of patients with other diagnoses on the same day", non_bd_same_day_count),
    ("Patients with lab AFTER BD diagnosis and non-BD diagnosis on same day", nonbd_diag_same_day_after),
    ("Total lab rows after BD diagnosis (within 2 years)", bd_labs_within_2yrs.shape[0]),
    ("Unique patients with labs in 2 years", bd_labs_within_2yrs['Patient_ID'].nunique()),
    ("Patients with MULTIPLE entries for SAME test on SAME day", patients_with_dup_tests),
    ("Patients with comorbidity (non-BD diagnosis on/after BD)", bd_labs_within_2yrs['Comorbidity'].sum())
]
wrg.print_summary_stats(summary_stats)

All patients have BD as their first-ever diagnosis: True
Total BD patients (first-ever diagnosis): 378
Patients with ANY lab results: 218
Patients with labs BEFORE diagnosis: 22
Patients with labs AFTER diagnosis: 214
Patients with labs BOTH before and after diagnosis: 18
Number of patients with other diagnoses on the same day: 105
Patients with lab AFTER BD diagnosis and non-BD diagnosis on same day: 71
Total lab rows after BD diagnosis (within 2 years): 319
Unique patients with labs in 2 years: 154
Patients with MULTIPLE entries for SAME test on SAME day: 0
Patients with comorbidity (non-BD diagnosis on/after BD): 308
